<div align="center">

<img src="https://raw.githubusercontent.com/YousufAKhan/yakRNA/main/mascot.png" width="180">

# YAK RNA

### A Deep Learning RNA Sequence Designer

[![GitHub](https://img.shields.io/github/stars/YousufAKhan/yakRNA?style=for-the-badge&logo=github)](https://github.com/YousufAKhan/yakRNA)
[![License](https://img.shields.io/badge/License-MIT-blue?style=for-the-badge)](https://github.com/YousufAKhan/yakRNA/blob/main/LICENSE)
[![Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=for-the-badge&logo=googlecolab)](https://colab.research.google.com/github/YousufAKhan/yakRNA/blob/main/yakRNA_colab.ipynb)

---

**Generate RNA sequences conditioned on:**

🔬 **Secondary Structure** · 🧬 **Consensus Sequence** · 🏷️ **GO Terms**

---

</div>

### 📋 Quick Start

1. ▶️ Run **Setup** to install dependencies (~2 min)
2. 🔍 Use **GO Term Search** to find functional annotations (optional)
3. ⚙️ Configure **Input Parameters** 
4. 🧬 Click **Generate** to design sequences
5. 📥 **Download** your results as FASTA

In [ ]:
#@title 🚀 **Step 1: Setup** { display-mode: "form" }
#@markdown Run this cell to install YAK RNA and download the model checkpoint.
#@markdown 
#@markdown ⏱️ **First run takes ~2 minutes** (cached afterwards)

import os
import sys
from IPython.display import display, HTML, Image, clear_output
import time

def styled_print(message, style="info"):
    """Print with HTML styling"""
    colors = {
        "info": "#2196F3",
        "success": "#4CAF50", 
        "warning": "#FF9800",
        "error": "#f44336"
    }
    icons = {
        "info": "ℹ️",
        "success": "✅",
        "warning": "⚠️",
        "error": "❌"
    }
    color = colors.get(style, colors["info"])
    icon = icons.get(style, "")
    display(HTML(f'<p style="color:{color}; font-size:14px; margin:5px 0;">{icon} {message}</p>'))

# Show progress
display(HTML('<h3>🔧 Setting up YAK RNA...</h3>'))

# Clone repo if not already done
if not os.path.exists("yakRNA"):
    styled_print("Cloning YAK RNA repository...", "info")
    
    # Try public clone first, fall back to authenticated if private
    result = os.system("git clone -q https://github.com/YousufAKhan/yakRNA.git 2>/dev/null")
    
    if result != 0:
        display(HTML('''
        <div style="background:#fff3cd; padding:15px; border-radius:8px; margin:10px 0;">
            <b>🔐 Repository is private</b><br>
            Please enter your GitHub Personal Access Token.<br>
            <small>Create one at: <a href="https://github.com/settings/tokens" target="_blank">github.com/settings/tokens</a> with 'repo' scope</small>
        </div>
        '''))
        from getpass import getpass
        token = getpass("GitHub Token: ")
        os.system(f"git clone -q https://{token}@github.com/YousufAKhan/yakRNA.git")
        del token
    
    styled_print("Repository cloned", "success")
else:
    styled_print("Repository already exists", "success")

os.chdir("yakRNA")
sys.path.append("inference")

# Display mascot centered
if os.path.exists("mascot.png"):
    display(HTML('<div style="text-align:center; margin:20px 0;">'))
    display(Image("mascot.png", width=180))
    display(HTML('</div>'))

# Install dependencies
styled_print("Installing dependencies...", "info")
!pip install -q torch transformers accelerate biopython pyyaml h5py huggingface_hub 2>/dev/null

# Download model checkpoint from Hugging Face
if not os.path.exists("checkpoint.pt"):
    styled_print("Downloading model checkpoint from Hugging Face (1.4 GB)...", "info")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="MasterYster/yakRNA-Design",
        filename="yakRNA_110M.pt",
        local_dir="."
    )
    os.rename("yakRNA_110M.pt", "checkpoint.pt")
    styled_print("Model checkpoint downloaded", "success")
else:
    styled_print("Model checkpoint found", "success")

# Final success message
display(HTML('''
<div style="background:#d4edda; padding:20px; border-radius:10px; margin-top:20px; text-align:center;">
    <h3 style="color:#155724; margin:0;">✅ Setup Complete!</h3>
    <p style="color:#155724; margin:10px 0 0 0;">You can now configure your parameters and generate sequences.</p>
</div>
'''))

In [ ]:
#@title 🔍 **Step 2: GO Term Search** *(Optional)* { display-mode: "form" }
#@markdown Search for Gene Ontology terms by keyword to condition your RNA generation.
#@markdown 
#@markdown **Supported:** 280 GO terms | **Max per sequence:** 12

import json
import os
from IPython.display import display, HTML

search_query = "ribosome"  #@param {type:"string"}
#@markdown 💡 **Try:** `ribosome`, `translation`, `splicing`, `viral`, `binding`, `regulation`

# Load GO term descriptions
go_file = "training_data/processed/vocabulary_analysis/go_term_descriptions.json"
if os.path.exists(go_file):
    with open(go_file) as f:
        go_terms_db = json.load(f)
    
    # Search
    query_lower = search_query.lower()
    matches = [(go_id, desc) for go_id, desc in go_terms_db.items() 
               if query_lower in desc.lower() or query_lower in go_id.lower()]
    
    if matches:
        # Build HTML table
        rows = ""
        for go_id, desc in sorted(matches, key=lambda x: x[1]):
            rows += f'<tr><td style="font-family:monospace; color:#1565C0; padding:8px;"><b>{go_id}</b></td><td style="padding:8px;">{desc}</td></tr>'
        
        display(HTML(f'''
        <div style="margin:15px 0;">
            <h4 style="color:#2E7D32;">🎯 Found {len(matches)} GO terms matching "{search_query}"</h4>
            <div style="max-height:300px; overflow-y:auto; border:1px solid #ddd; border-radius:8px;">
                <table style="width:100%; border-collapse:collapse;">
                    <thead style="background:#f5f5f5; position:sticky; top:0;">
                        <tr>
                            <th style="padding:10px; text-align:left; border-bottom:2px solid #ddd;">GO ID</th>
                            <th style="padding:10px; text-align:left; border-bottom:2px solid #ddd;">Description</th>
                        </tr>
                    </thead>
                    <tbody>{rows}</tbody>
                </table>
            </div>
            <p style="color:#666; margin-top:10px; font-size:13px;">
                📋 <b>Copy GO IDs</b> to the Input cell below. Use commas to separate multiple terms.
            </p>
        </div>
        '''))
    else:
        display(HTML(f'''
        <div style="background:#fff3cd; padding:15px; border-radius:8px; margin:10px 0;">
            <b>⚠️ No GO terms found matching "{search_query}"</b><br>
            <span style="color:#666;">Try broader terms like: RNA, binding, regulation, viral, ribosome, translation</span>
        </div>
        '''))
else:
    display(HTML('''
    <div style="background:#f8d7da; padding:15px; border-radius:8px;">
        <b>❌ GO term database not found</b><br>
        Please run the Setup cell first.
    </div>
    '''))

In [ ]:
#@title ⚙️ **Step 3: Input Parameters** { display-mode: "form" }
#@markdown Configure your RNA generation. **Limits:** Max length: 636 | Max GO terms: 12

from IPython.display import display, HTML
import re

#@markdown ---
#@markdown ### 🎯 Quick Presets
preset = "Custom"  #@param ["Custom", "Small hairpin (12 nt)", "Frameshift Pseudoknot (81 nt)"]

#@markdown ---
#@markdown ### 📊 Generation Settings
num_sequences = 5  #@param {type:"integer"}
temperature = 1.0  #@param {type:"slider", min:0.1, max:2.0, step:0.1}
#@markdown *Lower temperature = more conservative, Higher = more diverse*

#@markdown ---
#@markdown ### 🔬 Conditioning Modalities
#@markdown *Leave empty to skip a modality*

secondary_structure = ""  #@param {type:"string"}
#@markdown `((((....))))` or `:::<<<___>>>:::`

consensus_sequence = ""  #@param {type:"string"}
#@markdown `GAGUaaGGGG` *(uppercase=conserved, lowercase=variable)*

go_terms = ""  #@param {type:"string"}
#@markdown `GO:0075523` or `GO:0003676,GO:0005515`

#@markdown ---
#@markdown ### 🧩 Partial Sequence / Infilling *(Optional)*
#@markdown Provide known nucleotides and use `*` for positions to generate.
#@markdown If combined with secondary structure or consensus, lengths must match.

partial_sequence = ""  #@param {type:"string"}
#@markdown Example: `AUGC****CGAU` — fixed positions are kept, `*` positions are generated

sequence_length = 80  #@param {type:"integer"}
#@markdown *Only used if no structure, consensus, or partial sequence is provided*

#@markdown ---
#@markdown ### 🔧 Advanced Options
constraint_set = "canonical"  #@param ["canonical", "strict", "canonical+sheared", "canonical+common", "permissive", "unconstrained"]
#@markdown Base-pairing rules: `strict` (AU,GC) → `canonical` (+GU) → `permissive` → `unconstrained` (all pairs allowed)

output_filename = "generated_sequences.fasta"  #@param {type:"string"}

# Apply presets
if preset == "Small hairpin (12 nt)":
    secondary_structure = "((((....))))"
    consensus_sequence = ""
    go_terms = ""
    partial_sequence = ""
    sequence_length = 12
elif preset == "Frameshift Pseudoknot (81 nt)":
    secondary_structure = ":::::::<<<<<<<<<-:::--[[[[[-->>>>>>>>><<<<<<<<<<_________>>>->>>>>>>::::]]]]]::::"
    consensus_sequence = "GAGUaaGGGGuuCuAGU...gcaGCcCgcCUaGaaCCCUGcgacacuGGuucuaaaaCagAugucgUuuuaAGgGCuUUUG"
    go_terms = "GO:0075523"
    partial_sequence = ""
    constraint_set = "canonical"
    sequence_length = 81

# Validation
errors = []
warnings = []

# Validate structure syntax
if secondary_structure:
    # Check for balanced brackets
    brackets = {'(': ')', '<': '>', '[': ']', '{': '}'}
    stacks = {b: [] for b in brackets.keys()}
    for i, char in enumerate(secondary_structure):
        if char in brackets:
            stacks[char].append(i)
        elif char in brackets.values():
            opening = [k for k, v in brackets.items() if v == char][0]
            if not stacks[opening]:
                errors.append(f"Unmatched '{char}' at position {i+1}")
            else:
                stacks[opening].pop()
    for opening, stack in stacks.items():
        if stack:
            errors.append(f"Unmatched '{opening}' at positions {[s+1 for s in stack]}")
    
    # Check length
    if len(secondary_structure) > 636:
        errors.append(f"Structure length ({len(secondary_structure)}) exceeds max (636)")

# Validate consensus matches structure length
if secondary_structure and consensus_sequence:
    if len(consensus_sequence) != len(secondary_structure):
        errors.append(f"Consensus length ({len(consensus_sequence)}) must match structure length ({len(secondary_structure)})")

# Validate partial sequence
if partial_sequence:
    partial_upper = partial_sequence.upper()
    valid_chars = set('AUGC*')
    invalid_chars = set(partial_upper) - valid_chars
    if invalid_chars:
        errors.append(f"Partial sequence contains invalid characters: {', '.join(sorted(invalid_chars))}. Use A, U, G, C, or * for masked positions.")
    elif '*' not in partial_upper:
        warnings.append("Partial sequence has no masked (*) positions — all positions are fixed. Did you mean to use the consensus sequence field instead?")
    else:
        if len(partial_upper) > 636:
            errors.append(f"Partial sequence length ({len(partial_upper)}) exceeds max (636)")
        if secondary_structure and len(partial_upper) != len(secondary_structure):
            errors.append(f"Partial sequence length ({len(partial_upper)}) must match secondary structure length ({len(secondary_structure)})")
        if consensus_sequence and len(partial_upper) != len(consensus_sequence):
            errors.append(f"Partial sequence length ({len(partial_upper)}) must match consensus sequence length ({len(consensus_sequence)})")
    partial_sequence = partial_upper  # normalize to uppercase

# Validate sequence length
if sequence_length > 636:
    warnings.append(f"Sequence length ({sequence_length}) exceeds max (636). Will be capped.")
    sequence_length = 636

# Validate GO terms count
if go_terms:
    go_list = [g.strip() for g in go_terms.split(',') if g.strip()]
    if len(go_list) > 12:
        warnings.append(f"{len(go_list)} GO terms provided, but max is 12. Only first 12 will be used.")
    # Validate GO term format
    for gt in go_list:
        if not re.match(r'^GO:\d{7}$', gt):
            errors.append(f"Invalid GO term format: '{gt}'. Expected format: GO:0000000")

# Store parameters
params = {
    'num_sequences': num_sequences,
    'temperature': temperature,
    'secondary_structure': secondary_structure,
    'consensus': consensus_sequence,
    'go_terms': go_terms,
    'partial_sequence': partial_sequence,
    'length': sequence_length,
    'constraint_set': constraint_set,
    'output': output_filename
}

# Display validation results
if errors:
    error_html = "<br>".join([f"• {e}" for e in errors])
    display(HTML(f'''
    <div style="background:#f8d7da; padding:15px; border-radius:8px; margin:10px 0; border-left:4px solid #dc3545;">
        <b style="color:#721c24;">❌ Validation Errors</b><br>
        <span style="color:#721c24;">{error_html}</span>
    </div>
    '''))
elif warnings:
    warning_html = "<br>".join([f"• {w}" for w in warnings])
    display(HTML(f'''
    <div style="background:#fff3cd; padding:15px; border-radius:8px; margin:10px 0; border-left:4px solid #ffc107;">
        <b style="color:#856404;">⚠️ Warnings</b><br>
        <span style="color:#856404;">{warning_html}</span>
    </div>
    '''))

# Display summary
if not errors:
    summary_items = [f"<b>Sequences:</b> {num_sequences}", f"<b>Temperature:</b> {temperature}"]
    if secondary_structure:
        struct_preview = secondary_structure[:40] + "..." if len(secondary_structure) > 40 else secondary_structure
        summary_items.append(f"<b>Structure:</b> <code>{struct_preview}</code> ({len(secondary_structure)} nt)")
    if consensus_sequence:
        cons_preview = consensus_sequence[:40] + "..." if len(consensus_sequence) > 40 else consensus_sequence
        summary_items.append(f"<b>Consensus:</b> <code>{cons_preview}</code>")
    if go_terms:
        summary_items.append(f"<b>GO Terms:</b> <code>{go_terms}</code>")
    if partial_sequence:
        num_masked = partial_sequence.count('*')
        num_fixed = len(partial_sequence) - num_masked
        ps_preview = partial_sequence[:40] + "..." if len(partial_sequence) > 40 else partial_sequence
        summary_items.append(f"<b>Partial Sequence:</b> <code>{ps_preview}</code> ({num_fixed} fixed, {num_masked} masked)")
    if not secondary_structure and not consensus_sequence and not partial_sequence:
        summary_items.append(f"<b>Length:</b> {sequence_length} nt")
    
    display(HTML(f'''
    <div style="background:#d4edda; padding:15px; border-radius:8px; margin:10px 0; border-left:4px solid #28a745;">
        <b style="color:#155724;">✅ Parameters Ready</b><br>
        <div style="color:#155724; margin-top:8px;">{"<br>".join(summary_items)}</div>
    </div>
    '''))

In [ ]:
#@title 🧬 **Step 4: Generate Sequences** { display-mode: "form" }
#@markdown Click the play button to generate RNA sequences with your parameters.

import subprocess
import os
import sys
import time
from IPython.display import display, HTML, clear_output
sys.path.append("inference")

# Check for validation errors
if 'errors' in dir() and errors:
    display(HTML('''
    <div style="background:#f8d7da; padding:20px; border-radius:10px; text-align:center;">
        <h3 style="color:#721c24; margin:0;">❌ Cannot Generate</h3>
        <p style="color:#721c24;">Please fix the validation errors in the Input cell first.</p>
    </div>
    '''))
else:
    # Show loading animation
    display(HTML('''
    <div id="loading" style="text-align:center; padding:30px;">
        <div style="display:inline-block; width:50px; height:50px; border:4px solid #f3f3f3; border-top:4px solid #3498db; border-radius:50%; animation:spin 1s linear infinite;"></div>
        <p style="color:#666; margin-top:15px; font-size:16px;">🧬 Generating RNA sequences...</p>
        <p style="color:#999; font-size:13px;">This may take a minute depending on sequence length</p>
    </div>
    <style>@keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }</style>
    '''))
    
    # Build command
    cmd = [
        "python", "inference/rna_sequence_generator.py",
        "--config", "configs/inference.yaml",
        "--checkpoint", "checkpoint.pt",
        "--num_sequences", str(params['num_sequences']),
        "--temperature", str(params['temperature']),
        "--fasta_output", params['output']
    ]
    
    if params.get('partial_sequence'):
        cmd.extend(["--masked_sequence", params['partial_sequence']])
    
    if params['secondary_structure']:
        cmd.extend(["--secondary_structure", params['secondary_structure']])
        cmd.extend(["--constraint_set", params['constraint_set']])
    elif not params.get('partial_sequence') and params['length']:
        cmd.extend(["--length", str(params['length'])])
    
    if params['consensus']:
        cmd.extend(["--consensus", params['consensus']])
    
    if params['go_terms']:
        cmd.extend(["--go_terms", params['go_terms']])
    
    # Run generation
    start_time = time.time()
    result = subprocess.run(cmd, text=True, capture_output=True)
    elapsed = time.time() - start_time
    
    clear_output(wait=True)
    
    if result.returncode == 0 and os.path.exists(params['output']):
        # Parse sequences from FASTA
        sequences = []
        headers = []
        with open(params['output'], 'r') as f:
            seq = ""
            header = ""
            for line in f:
                if line.startswith('>'):
                    if seq:
                        sequences.append(seq)
                        headers.append(header)
                    header = line.strip()[1:]
                    seq = ""
                else:
                    seq += line.strip()
            if seq:
                sequences.append(seq)
                headers.append(header)
        
        # Success header
        display(HTML(f'''
        <div style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding:25px; border-radius:12px; margin-bottom:20px; text-align:center; color:white;">
            <h2 style="margin:0; font-size:24px;">🎉 Generation Complete!</h2>
            <p style="margin:10px 0 0 0; opacity:0.9;">{len(sequences)} sequences generated in {elapsed:.1f}s</p>
        </div>
        '''))
        
        # Display each sequence beautifully
        partial = params.get('partial_sequence', '')
        for i, (seq, header) in enumerate(zip(sequences, headers), 1):
            # Build sequence display with coloring.
            # Fixed positions (from partial sequence template) shown in gray with dotted underline.
            # Generated positions shown in standard nucleotide colors.
            colored_seq = ""
            colors = {'A': '#e74c3c', 'U': '#3498db', 'G': '#2ecc71', 'C': '#f39c12'}
            for idx, nt in enumerate(seq):
                if partial and idx < len(partial) and partial[idx] != '*':
                    colored_seq += f'<span style="color:#888; text-decoration:underline; text-decoration-style:dotted;" title="Fixed position">{nt}</span>'
                else:
                    color = colors.get(nt, '#333')
                    colored_seq += f'<span style="color:{color}">{nt}</span>'
            
            # Structure alignment if available
            structure_html = ""
            if params['secondary_structure'] and len(params['secondary_structure']) == len(seq):
                structure_html = f'''
                <div style="font-family:monospace; font-size:12px; color:#666; margin-bottom:5px; letter-spacing:1px; word-break:break-all;">
                    {params['secondary_structure']}
                </div>
                '''
            
            # Partial sequence template if available
            partial_html = ""
            if partial and len(partial) == len(seq):
                partial_html = f'''
                <div style="font-family:monospace; font-size:12px; color:#aaa; margin-bottom:5px; letter-spacing:1px; word-break:break-all;" title="Template: * = generated positions">
                    {partial}
                </div>
                '''
            
            display(HTML(f'''
            <div style="background:#f8f9fa; border:1px solid #dee2e6; border-radius:8px; padding:15px; margin:10px 0;">
                <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
                    <span style="font-weight:bold; color:#495057;">Sequence {i}</span>
                    <span style="background:#e9ecef; padding:3px 10px; border-radius:12px; font-size:12px; color:#6c757d;">{len(seq)} nt</span>
                </div>
                {structure_html}
                {partial_html}
                <div style="font-family:'Courier New', monospace; font-size:13px; word-break:break-all; line-height:1.6; background:white; padding:10px; border-radius:4px; border:1px solid #e9ecef;">
                    {colored_seq}
                </div>
            </div>
            '''))
        
        # Constraint satisfaction if structure provided
        if params['secondary_structure']:
            display(HTML(f'''
            <div style="background:#e3f2fd; border:1px solid #90caf9; border-radius:8px; padding:15px; margin:20px 0;">
                <h3 style="margin:0 0 15px 0; color:#1565c0;">📊 Secondary Structure Constraint Satisfaction</h3>
            '''))
            
            from rna_sequence_generator import StructureConstrainedGenerator, PAIR_SETS
            
            constraint_checker = StructureConstrainedGenerator(
                vocabulary=None,
                allowed_pairs=PAIR_SETS.get(params['constraint_set'], PAIR_SETS['canonical'])
            )
            
            total_satisfaction = 0
            results_html = ""
            for i, seq in enumerate(sequences, 1):
                satisfaction = constraint_checker.calculate_constraint_satisfaction(
                    seq, params['secondary_structure']
                )
                is_valid, violations = constraint_checker.check_constraints(
                    seq, params['secondary_structure']
                )
                total_satisfaction += satisfaction
                
                if is_valid:
                    status_color = "#28a745"
                    status_icon = "✅"
                    status_text = "All constraints satisfied"
                else:
                    status_color = "#dc3545"
                    status_icon = "❌"
                    status_text = f"{len(violations)} violation(s)"
                
                bar_width = int(satisfaction * 100)
                results_html += f'''
                <div style="display:flex; align-items:center; margin:8px 0;">
                    <span style="width:80px; color:#495057;">Seq {i}:</span>
                    <div style="flex:1; background:#e9ecef; border-radius:4px; height:20px; margin:0 10px; overflow:hidden;">
                        <div style="width:{bar_width}%; background:{status_color}; height:100%; border-radius:4px;"></div>
                    </div>
                    <span style="width:60px; text-align:right; font-weight:bold; color:{status_color};">{satisfaction:.1%}</span>
                    <span style="width:30px; text-align:center;">{status_icon}</span>
                </div>
                '''
            
            avg_satisfaction = total_satisfaction / len(sequences) if sequences else 0
            display(HTML(f'''
                {results_html}
                <div style="margin-top:15px; padding-top:15px; border-top:1px solid #90caf9;">
                    <span style="font-weight:bold; color:#1565c0;">Average: {avg_satisfaction:.1%}</span>
                    <span style="color:#666; margin-left:20px;">Constraint set: <code>{params['constraint_set']}</code></span>
                </div>
            </div>
            '''))
        
        # Legend for partial sequence coloring
        if params.get('partial_sequence'):
            display(HTML('''
            <div style="margin:10px 0 5px 0; color:#666; font-size:13px;">
                <span style="color:#888; text-decoration:underline; text-decoration-style:dotted;">dotted underline</span> = fixed positions &nbsp;·&nbsp; <span style="color:#e74c3c;">A</span><span style="color:#3498db;">U</span><span style="color:#2ecc71;">G</span><span style="color:#f39c12;">C</span> colors = generated positions
            </div>
            '''))
        
        # Copy button hint
        display(HTML('''
        <div style="text-align:center; margin-top:20px; color:#666;">
            <p>💡 <b>Tip:</b> Use the Download button below to save sequences as FASTA</p>
        </div>
        '''))
        
    else:
        # Error display
        display(HTML(f'''
        <div style="background:#f8d7da; padding:20px; border-radius:10px; text-align:center;">
            <h3 style="color:#721c24; margin:0;">❌ Generation Failed</h3>
            <p style="color:#721c24;">Check the error details below.</p>
        </div>
        '''))
        if result.stderr:
            # Filter and show relevant errors
            stderr_lines = [l for l in result.stderr.split('\n') 
                          if l and not any(x in l for x in ['UserWarning', 'FutureWarning', 'weights_only'])]
            if stderr_lines:
                display(HTML(f'''
                <pre style="background:#f5f5f5; padding:15px; border-radius:8px; overflow-x:auto; font-size:12px;">
{chr(10).join(stderr_lines[-15:])}
                </pre>
                '''))

In [ ]:
#@title 📥 **Step 5: Download Results** { display-mode: "form" }
#@markdown Download your generated sequences as a FASTA file.

from google.colab import files
import os
from IPython.display import display, HTML

if os.path.exists(params['output']):
    # Get file size
    file_size = os.path.getsize(params['output'])
    size_str = f"{file_size} bytes" if file_size < 1024 else f"{file_size/1024:.1f} KB"
    
    # Count sequences
    with open(params['output'], 'r') as f:
        seq_count = sum(1 for line in f if line.startswith('>'))
    
    display(HTML(f'''
    <div style="background:#f8f9fa; border:2px dashed #dee2e6; border-radius:12px; padding:30px; text-align:center; margin:10px 0;">
        <div style="font-size:48px; margin-bottom:15px;">📄</div>
        <h3 style="margin:0; color:#495057;">{params['output']}</h3>
        <p style="color:#6c757d; margin:10px 0;">{seq_count} sequences · {size_str}</p>
    </div>
    '''))
    
    # Trigger download
    files.download(params['output'])
    
    display(HTML('''
    <div style="background:#d4edda; padding:15px; border-radius:8px; margin-top:15px; text-align:center;">
        <span style="color:#155724;">✅ Download started! Check your browser's download folder.</span>
    </div>
    '''))
else:
    display(HTML('''
    <div style="background:#f8d7da; padding:20px; border-radius:10px; text-align:center;">
        <h3 style="color:#721c24; margin:0;">❌ No Output File Found</h3>
        <p style="color:#721c24;">Please run the Generate cell first.</p>
    </div>
    '''))

---

<div align="center">

## 📖 Reference Guide

</div>

### Secondary Structure Notation

| Symbol | Meaning | Example |
|:------:|---------|---------|
| `(` `)` | Base pair (stem) | `(((...)))` |
| `<` `>` | Base pair (pseudoknot) | `<<<...>>>` |
| `[` `]` | Base pair (alternative) | `[[[...]]]` |
| `.` `:` | Unpaired | `....` |
| `_` `-` | Unpaired (loop/bulge) | `(((__)))` |

### Consensus Sequence

| Format | Meaning |
|--------|---------|
| **Uppercase** `AUGC` | Conserved position (must match) |
| **Lowercase** `augc` | Variable position (preferred) |
| **Dot** `.` | Any nucleotide |

### Base-Pairing Constraint Sets

| Set | Allowed Pairs | Use Case |
|-----|---------------|----------|
| `strict` | A:U, G:C | Watson-Crick only |
| `canonical` | + G:U | Standard RNA (default) |
| `canonical+sheared` | + G:A | Include sheared pairs |
| `permissive` | + A:C, U:C | Maximum flexibility |
| `unconstrained` | All 16 pairs | No base-pair restrictions |

---

<div align="center">

## 📚 Citation

If you use YAK RNA in your research, please cite:

```bibtex
@software{yakrna2025,
  author = {Khan, Yousuf},
  title = {YAK RNA: A Deep Learning RNA Sequence Designer},
  year = {2025},
  url = {https://github.com/YousufAKhan/yakRNA}
}
```

---

<p style="color:#666;">
Made with 🦬 by <a href="https://github.com/YousufAKhan">Yousuf Khan</a>
</p>

</div>